# BETO Fine-tuning — ROCKTEC MIA 2026

Corre `02_scripts/12_beto_finetuning.py` (fine-tuning real de BETO, 5 clases) sobre GPU gratuita de Colab.

**Antes de correr:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → GPU (T4).
2. Asegúrate de haber hecho `git push` de tus cambios locales (incluyendo `02_scripts/12_beto_finetuning.py`) a `origin/main` — este notebook clona el repo desde GitHub, no ve tu filesystem local.

Si todavía no pusheaste, usa la celda opcional al final ("Alternativa: subir archivos manualmente") en vez del `git clone`.

## 1. Verificar GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA disponible:', torch.cuda.is_available())

name, memory.total [MiB]
Tesla T4, 15360 MiB
CUDA disponible: True


## 2. Clonar el repositorio

Requiere que el repo sea público (o que uses un token si es privado — reemplaza la URL por `https://<TOKEN>@github.com/LuisChica18/proyecto-mia-rocktec.git`).

In [6]:
REPO_URL = 'https://github.com/LuisChica18/proyecto-mia-rocktec.git'

import os
if os.path.isdir('proyecto-mia-rocktec'):
    %cd proyecto-mia-rocktec
    !git pull
else:
    !git clone $REPO_URL
    %cd proyecto-mia-rocktec

!git log --oneline -5

/content/proyecto-mia-rocktec
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 16 (delta 11), reused 16 (delta 11), pack-reused 0 (from 0)
Unpacking objects: 100% (16/16), 11.06 KiB | 2.21 MiB/s, done.
From https://github.com/LuisChica18/proyecto-mia-rocktec
   f8715ad..6957cd6  main       -> origin/main
Updating f8715ad..6957cd6
Fast-forward
 .gitattributes                                  |   1 +
 02_scripts/12_beto_finetuning.py                | 236 +++++++++++
 02_scripts/12_beto_finetuning_colab.ipynb       | 295 ++++++++++++++
 02_scripts/calcular_kappa.py                    | 496 ++++++++++++------------
 "05_documentacion/DISE\303\221O_MLOPS_FASE2.md" |  15 +-
 CHANGELOG.md                                    |  17 +
 README.md                                       | 384 +++++++++---------
 requirements.txt                                |  96 ++---
 8 files changed, 1052 insertions(

In [7]:
!pwd

/content/proyecto-mia-rocktec


In [7]:
!ls /content/proyecto-mia-rocktec

01_datos_crudos      04_anotaciones    CHANGELOG.md	     requirements.txt
02_scripts	     05_documentacion  proyecto-mia-rocktec
03_datos_procesados  06_resultados     README.md


In [8]:
!ls /content/proyecto-mia-rocktec/02_scripts

01_limpieza_datos_CORREGIDO.py	    08_buscar_candidatos_que_seg.py
01_limpieza_datos.py		    09_crear_holdout_set.py
02_consolidar_datos_CORREGIDO.py    10_shap_lime_explicabilidad.py
02_consolidar_datos.py		    11_beto_clasificador.py
03_validar_duplicados_CORREGIDO.py  12_beto_finetuning_colab.ipynb
03_validar_duplicados.py	    12_beto_finetuning.py
04_feature_engineering.py	    calcular_kappa.py
05_entrenar_modelos.py		    consolidar_4_bases.py
06_pipeline_completo.py		    generar_informe_docx.py
07_validacion_estadistica.py


## 3. Instalar dependencias

Colab ya trae `torch`, `pandas` y `scikit-learn` preinstalados y compatibles entre sí (con `cudf`/`google-colab`, que exigen `pandas<2.4`) — **no los reinstales ni los incluyas en el `pip install`**, o vas a romper esas dependencias (por ejemplo, forzar `pandas` a la 3.x rompe `google.colab.files.download` que usamos más abajo).

Solo necesitamos actualizar `transformers`/`accelerate`, y desinstalar `peft` (no lo usamos — este script hace fine-tuning completo, no LoRA — y a veces queda desincronizado con `accelerate`, dando el error `cannot import name 'clear_device_cache' from 'accelerate.utils.memory'`).

**Si ya corriste la celda anterior (con `pandas` incluido) y te salió el conflicto de versiones:** `Entorno de ejecución` → `Reiniciar sesión` para volver al entorno limpio de Colab, y luego corre esta celda ya corregida.

**Después de correr esta celda: `Entorno de ejecución` → `Reiniciar sesión`**, y luego continúa desde el paso 4 (no hace falta repetir el clonado).

In [ ]:
%pip install -q -U transformers accelerate

## 4. Correr el fine-tuning

Usa `04_anotaciones/dataset_consenso_final.csv` (ya está en el repo clonado). Con GPU T4, 5 épocas sobre ~1,050 registros de entrenamiento debería tomar unos pocos minutos.

In [9]:
!python 02_scripts/12_beto_finetuning.py

BETO FINE-TUNING — ROCKTEC MIA 2026
[1/5] Cargando dataset...
  ✓ 1312 registros, 5 clases
    INF: 870
    COT: 289
    CUR: 61
    TEC: 51
    VEN: 41
[2/5] GPU disponible: True (Tesla T4)
  Train: 891  Val: 158  Test: 263
[3/5] Cargando BETO + tokenizer...
config.json: 100% 648/648 [00:00<00:00, 4.04MB/s]
tokenizer_config.json: 100% 364/364 [00:00<00:00, 2.37MB/s]
vocab.txt: 100% 242k/242k [00:00<00:00, 84.6MB/s]
tokenizer.json: 100% 480k/480k [00:00<00:00, 108MB/s]
special_tokens_map.json: 100% 134/134 [00:00<00:00, 915kB/s]

pytorch_model.bin: downloading bytes:  64% 282M/440M [00:03<00:01, 149MB/s, 24.0MB/s  ]  
pytorch_model.bin: downloading bytes:  89% 392M/440M [00:03<00:00, 222MB/s, 31.4MB/s  ]
pytorch_model.bin: reconstructing file:  76% 335M/440M [00:03<00:01, 93.4MB/s, 24.9MB/s  ]
pytorch_model.bin: downloading bytes: 100% 417M/417M [00:03<00:00, 108MB/s, 35.8MB/s  ] ] 
pytorch_model.bin: reconstructing file: 100% 440M/440M [00:03<00:00, 114MB/s, 39.0MB/s  ]
Loading weight

## 5. Ver resultados

In [10]:
!echo '--- comparacion_tfidf_vs_beto_finetuned.txt ---'
!cat 06_resultados/beto/comparacion_tfidf_vs_beto_finetuned.txt
!echo '--- reporte_beto_finetuned.txt ---'
!cat 06_resultados/beto/reporte_beto_finetuned.txt

--- comparacion_tfidf_vs_beto_finetuned.txt ---

COMPARACIÓN TF-IDF vs BETO (embeddings) vs BETO (fine-tuned) — ROCKTEC MIA 2026
Fecha: 2026-07-25 16:38:26
Dataset: consenso humano (1312 registros, 5 clases)
Split test: 80/20 estratificado, random_state=42 (mismo split que script 11)

RESULTADOS:
  TF-IDF + LR (full dataset):              F1-macro = 0.7516  ✅ META ≥ 0.75
  BETO embeddings + LR (sin fine-tuning):  F1-macro = 0.6370
  BETO fine-tuned (5 épocas):              F1-macro = 0.8552  ✅ META ≥ 0.75

INTERPRETACIÓN:
  - BETO fine-tuned supera a TF-IDF + LR
  - BETO fine-tuned supera a BETO sin fine-tuning (como era de esperar,
    el ajuste de pesos sobre el dominio de construcción/concreto decorativo
    ecuatoriano debería mejorar sobre los embeddings genéricos)

CONCLUSIÓN:
  El fine-tuning de BETO iguala o mejora el F1-macro sobre TF-IDF+LR; evaluar
  si el costo de infraestructura GPU se justifica para producción.

--- reporte_beto_finetuned.txt ---
              precision  

## 6. Descargar resultados a tu máquina

Empaqueta los reportes + el checkpoint del modelo y los descarga como zip. El checkpoint (`beto_finetuned_best/`) puede pesar ~440MB — si solo te interesan los reportes de texto, comenta la línea del checkpoint.

In [11]:
!zip -r resultados_beto_finetuning.zip 06_resultados/beto/reporte_beto_finetuned.txt 06_resultados/beto/comparacion_tfidf_vs_beto_finetuned.txt 06_resultados/modelos/beto_finetuned_best

from google.colab import files
files.download('resultados_beto_finetuning.zip')

  adding: 06_resultados/beto/reporte_beto_finetuned.txt (deflated 64%)
  adding: 06_resultados/beto/comparacion_tfidf_vs_beto_finetuned.txt (deflated 54%)
  adding: 06_resultados/modelos/beto_finetuned_best/ (stored 0%)
  adding: 06_resultados/modelos/beto_finetuned_best/config.json (deflated 53%)
  adding: 06_resultados/modelos/beto_finetuned_best/tokenizer_config.json (deflated 47%)
  adding: 06_resultados/modelos/beto_finetuned_best/model.safetensors (deflated 7%)
  adding: 06_resultados/modelos/beto_finetuned_best/tokenizer.json (deflated 71%)
  adding: 06_resultados/modelos/beto_finetuned_best/training_args.bin (deflated 53%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Después de descargar, descomprime el zip en la raíz de tu copia local del repo (respetando las rutas `06_resultados/...`) y haz el commit desde VS Code como de costumbre.

In [12]:
!ls -lh /content

total 8.0K
drwxr-xr-x 10 root root 4.0K Jul 25 16:40 proyecto-mia-rocktec
drwxr-xr-x  1 root root 4.0K Jun  4 13:39 sample_data


In [13]:
!ls -lh resultados_beto_finetuning.zip

-rw-r--r-- 1 root root 389M Jul 25 16:40 resultados_beto_finetuning.zip


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
!cp resultados_beto_finetuning.zip "/content/drive/MyDrive/"